<style>
    .input_area, .output_area, pre, code {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
    }
</style>

Download NPA Data from Charlotte's ArcGIS API and join it with Redlining map using geospatial data

In [29]:
# import libraries

import geopandas as gpd
import pandas as pd
import requests
from functools import reduce
from sklearn.impute import SimpleImputer
import numpy as np
import sqlite3


#### 'clt_holc'  
Redlining data from 'Mapping Inequality' by University of Richmond: https://dsl.richmond.edu/panorama/redlining  
Contains historic Home Owners' Loan Corportation grading for US Cities from 1935 - 1940

In [19]:
# Load data
map = gpd.read_file("https://raw.githubusercontent.com/coffeeNqueries/DTSC2302-Project3/refs/heads/main/data/raw_data/mappinginequality.json")

# Filter to Charlotte, NC
clt_holc = map[(map["city"] == "Charlotte") & (map["state"] == "NC")]

# label geometry as latitude and longitude. Then set to meters for accurate area calculations.
clt_holc = clt_holc.set_crs("EPSG:4326").to_crs("EPSG:3857")

# Select relevant columns
clt_holc = clt_holc[["area_id", "city", "state", "category", "grade", "residential", "commercial", "industrial", "geometry"]]

print("====== Charlotte HOLC Data ======")
print(f"Shape: {clt_holc.shape}")
print(clt_holc["grade"].value_counts())

display(clt_holc.head())

====== Charlotte HOLC Data ======
Shape: (35, 9)
grade
C    13
D    11
B     7
A     3
Name: count, dtype: int64


,area_id,city,state,category,grade,residential,commercial,industrial,geometry
5296,263,Charlotte,NC,Best,A,True,False,False,"MULTIPOLYGON (((-8994519.121 4194520.381, -899..."
5297,278,Charlotte,NC,Best,A,True,False,False,"MULTIPOLYGON (((-8996348.101 4190806.229, -899..."
5298,274,Charlotte,NC,Best,A,True,False,False,"MULTIPOLYGON (((-8999657.629 4192992.877, -899..."
5299,262,Charlotte,NC,Still Desirable,B,True,False,False,"MULTIPOLYGON (((-8994706.138 4195238.557, -899..."
5300,265,Charlotte,NC,Still Desirable,B,True,False,False,"MULTIPOLYGON (((-8995683.523 4193862.206, -899..."


#### 'npa'   
Data from City of Charlotte, NC ArcGIS REST API.   
Contains NPA boundaries and features.  
"Neighborhood Profile Areas used in the Housing Locational Scoring Tool. This data supports the methods behind generating scores for affordable housing investments." 
Last updated Dec. 2024

In [20]:
# Load data from ArcGIS REST API
npa = gpd.read_file("https://gis.charlottenc.gov/arcgis/rest/services/HNS/NPA_HLT/FeatureServer/0/query?where=1%3D1&outFields=*&f=geojson")

# Most recent record for each NPA
npa = npa.sort_values("Vintage").groupby("NPAs_NPA").last().reset_index()

# Label CRS as lat/long, then set to meters to match HOLC
npa = npa.set_geometry("geometry").set_crs("EPSG:4326").to_crs("EPSG:3857")

print("\n=== NPA ===")
print(f"Shape: {npa.shape}")
print(f"Unique NPAs: {npa['NPAs_NPA'].nunique()}\n")

npa.info()


=== NPA ===
Shape: (463, 27)
Unique NPAs: 463

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 463 entries, 0 to 462
Data columns (total 27 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   NPAs_NPA                         463 non-null    int32   
 1   OBJECTID                         463 non-null    int32   
 2   NPAs_LandArea                    463 non-null    int32   
 3   NPA_Lookup__NPA                  463 non-null    float64 
 4   NPA_Lookup__Access_Score         463 non-null    float64 
 5   NPA_Lookup__Jobs_Accessible_by_  463 non-null    float64 
 6   NPA_Lookup__Jobs_Accessible_by1  463 non-null    float64 
 7   NPA_Lookup__Change_Score         463 non-null    float64 
 8   NPA_Lookup__Median_Household_In  463 non-null    int32   
 9   NPA_Lookup__Median_Household__1  463 non-null    float64 
 10  NPA_Lookup__Permits_per_1_000    463 non-null    float64 
 11  NPA_Lookup__Per

In [21]:
# Rename relevant features and drop the rest
npa = npa.rename(columns = {
    "NPAs_NPA":                        "npa",
    "NPAs_LandArea":                   "land_area",
    "NPA_Lookup__Access_Score":        "transit_access_score",
    "NPA_Lookup__Jobs_Accessible_by_": "jobs_accessible_transit",
    "NPA_Lookup__Jobs_Accessible_by1": "jobs_accessible_car",
    "NPA_Lookup__Change_Score":        "neighborhood_change_score",
    "NPA_Lookup__Median_Household_In": "median_household_income",
    "NPA_Lookup__Permits_per_1_000":   "permits_per_1000",
    "NPA_Lookup__Median_Sales_Price":  "median_sales_price",
    "NPA_Lookup__Change_in_Sales_Pri": "sales_price_change",
    "FNS_2016":                        "food_nutrition_services_2016",
    "NPA_Lookup_pvrty_median_rate":    "poverty_rate_median"
})

npa = npa[[
    "npa",
    "land_area",
    "transit_access_score",
    "jobs_accessible_transit",
    "jobs_accessible_car",
    "neighborhood_change_score",
    "median_household_income",
    "permits_per_1000",
    "median_sales_price",
    "sales_price_change",
    "food_nutrition_services_2016",
    "poverty_rate_median",
    "geometry"
]]

display(npa.head())
npa.describe()

,npa,land_area,transit_access_score,jobs_accessible_transit,jobs_accessible_car,neighborhood_change_score,median_household_income,permits_per_1000,median_sales_price,sales_price_change,food_nutrition_services_2016,poverty_rate_median,geometry
0,2,411,5.56,1166805.0,130539.0,9.73,47500,10.657194,565000,0.419598,18.0,16,"POLYGON ((-8992560.48 4192789.436, -8992524.63..."
1,3,1156,7.55,1201401.0,234652.0,4.25,105303,2.497191,1079000,0.017925,3.0,16,"POLYGON ((-9000567.693 4190731.288, -9000524.6..."
2,4,332,4.46,1043532.0,88731.0,4.44,212045,7.281553,1714500,0.524000,0.0,16,"POLYGON ((-8994665.532 4185814.236, -8994702.4..."
3,5,167,5.87,1273789.0,131891.0,9.89,30547,5.797101,212500,0.221264,55.0,16,"POLYGON ((-9004438.162 4198414.07, -9004417.90..."
4,6,403,5.98,1265675.0,139226.0,9.97,32404,32.178218,534500,1.138000,44.0,16,"POLYGON ((-9003100.991 4195625.449, -9003089.7..."


,npa,land_area,transit_access_score,jobs_accessible_transit,jobs_accessible_car,neighborhood_change_score,median_household_income,permits_per_1000,median_sales_price,sales_price_change,food_nutrition_services_2016,poverty_rate_median
count,463.000000,463.000000,463.000000,4.630000e+02,463.000000,463.000000,463.000000,463.000000,4.630000e+02,463.000000,463.000000,463.000000
mean,236.369330,757.360691,3.584579,1.039543e+06,41497.114471,5.051231,56213.857451,10.768478,3.249903e+05,43.270244,14.593952,12.857451
std,136.232436,821.293029,1.508528,1.877299e+05,66875.179179,3.179041,45319.420470,28.870092,2.880957e+05,929.475416,14.755853,6.366272
min,2.000000,67.000000,0.710000,2.599010e+05,0.000000,0.000000,0.000000,0.000000,0.000000e+00,-0.388160,0.000000,0.000000
25%,119.500000,294.000000,2.675000,9.702080e+05,0.000000,2.780000,29389.000000,0.000000,0.000000e+00,0.000000,1.000000,16.000000
50%,236.000000,509.000000,3.180000,1.073222e+06,3775.000000,5.570000,51736.000000,1.830384,3.050000e+05,0.000000,11.000000,16.000000
75%,352.500000,910.500000,4.275000,1.185278e+06,65006.000000,7.660000,80680.500000,8.861367,4.400000e+05,0.113519,24.000000,16.000000
max,476.000000,8537.000000,9.320000,1.278433e+06,324651.000000,10.000000,250000.000000,363.636364,1.783750e+06,20000.000000,83.000000,17.000000


#### 'npa_holc'    
Assign HOLC grades by using an overlay between NPA boundaries and HOLC zones that covers the largest portion of each NPA. 

In [22]:
overlay = gpd.overlay(clt_holc[["grade", "category", "geometry"]], npa, how="intersection")

# calculate area of each intersection
overlay["overlap_area"] = overlay.geometry.area

npa_holc = (
    overlay.sort_values("overlap_area", ascending=False)
    .groupby("npa")
    .first()
    .reset_index()[["npa", "grade", "category"]]
    .rename(columns={"category": "grade_category"})
)

print("\n=== NPA-HOLC Overlay ===")
print(f"Shape: {npa_holc.shape}")
print(npa_holc["grade"].value_counts())

display(npa_holc.head())


=== NPA-HOLC Overlay ===
Shape: (45, 3)
grade
D    21
C    17
B     5
A     2
Name: count, dtype: int64


,npa,grade,grade_category
0,3,C,Definitely Declining
1,5,C,Definitely Declining
2,6,C,Definitely Declining
3,9,C,Definitely Declining
4,10,C,Definitely Declining


#### 'qol'
Quality of Life data from City of Charlotte ArcGIS REST API
Feature Types to ingest:  
- housing and target variables
- socioeconomic status
- demographics
- safety
- accessibility
- neighborhood characteristics

In [23]:
# load data
base_url = "https://gis.charlottenc.gov/arcgis/rest/services/QOL/Quality_of_Life_Latest_Profile_Data/MapServer"

# download each qol variable
def fetch_layer(layer_id):
    url = f"{base_url}/{layer_id}/query"
    params = {"where": "1=1", "outFields": "NPA, Val_Raw, Val_Norm", "f": "json", "resultRecordCount": 500} # there are 460+ NPAs
    response = requests.get(url, params=params)
    records = [f["attributes"] for f in response.json()["features"]]
    return pd.DataFrame(records)

# Housing and Target Variables
home_ownership = fetch_layer(62)  # owner occupied housing units / total occupied housing units
home_sales_price = fetch_layer(63) # Avg Home Sales Price
rental_cost = fetch_layer(69) # Median Gross Rent
rental_houses = fetch_layer(70) # detached, single-family rental housing units / total number of detached, single-family housing units

# Socioeconomic Status
household_income = fetch_layer(17) # Median Household Income
food_nutrition = fetch_layer(16) # pop. receiving FNS / total pop. Used as a proxy for poverty
health_insurance = fetch_layer(60) # pop. receiving Medicaid or NC Health Choice / total pop. Used as a proxy for poverty
employment = fetch_layer(15) # pop. aged 16 - 64 in labor force / total pop. aged 16 - 64

# Neighborhood Characteristics
housing_age = fetch_layer(64) # Avg age of housing units
code_violations = fetch_layer(66) # (number of housing code violations / total housing units) * 100
foreclosures = fetch_layer(72) # Number of single-fam, condo, townhome foreclosures / the number of single-fam, condo, townhomes for the fiscal year.
new_constructions = fetch_layer(73) # Number of res units permitted for new construction
housing_density = fetch_layer(67) # total housing units / total land area in acres

# Demographics
race_black = fetch_layer(7)
race_white = fetch_layer(9)
race_hispanic = fetch_layer(8)
race_asian = fetch_layer(6)
race_other = fetch_layer(5)
age_of_residents = fetch_layer(1) # Median age of residents


# Accessibility
transit_proximity = fetch_layer(87) # Number of housing units within half-mile of a transit stop / total number of housing units
financial_proximity = fetch_layer(19) # Housing units within half-mile of a federally insured bank or credit union / total number of housing units.
grocery_proximity = fetch_layer(56) # Number of housing units within half-mile of a grocery store / total number of housing units

# Safety
crime_property = fetch_layer(80) # (The number of property offenses in the fiscal year / the population) * 1,000
crime_violent = fetch_layer(81) # (The number of violent offenses in the fiscal year / the population) * 1,000

# Rename value columns to something meaningful
home_ownership       = home_ownership.rename(columns={"Val_Raw": "home_ownership",          "Val_Norm": "home_ownership_norm"})
home_sales_price     = home_sales_price.rename(columns={"Val_Raw": "home_sales_price",      "Val_Norm": "home_sales_price_norm"})
rental_cost          = rental_cost.rename(columns={"Val_Raw": "rental_cost",                "Val_Norm": "rental_cost_norm"})
rental_houses        = rental_houses.rename(columns={"Val_Raw": "rental_houses",            "Val_Norm": "rental_houses_norm"})

household_income     = household_income.rename(columns={"Val_Raw": "household_income",      "Val_Norm": "household_income_norm"})
food_nutrition       = food_nutrition.rename(columns={"Val_Raw": "food_nutrition",          "Val_Norm": "food_nutrition_norm"})
health_insurance     = health_insurance.rename(columns={"Val_Raw": "health_insurance",      "Val_Norm": "health_insurance_norm"})
employment           = employment.rename(columns={"Val_Raw": "employment",                  "Val_Norm": "employment_norm"})

housing_age          = housing_age.rename(columns={"Val_Raw": "housing_age",                "Val_Norm": "housing_age_norm"})
code_violations      = code_violations.rename(columns={"Val_Raw": "code_violations",        "Val_Norm": "code_violations_norm"})
foreclosures         = foreclosures.rename(columns={"Val_Raw": "foreclosures",              "Val_Norm": "foreclosures_norm"})
new_constructions    = new_constructions.rename(columns={"Val_Raw": "new_constructions",    "Val_Norm": "new_constructions_norm"})
housing_density      = housing_density.rename(columns={"Val_Raw": "housing_density",        "Val_Norm": "housing_density_norm"})

age_of_residents     = age_of_residents.rename(columns={"Val_Raw": "age_of_residents",      "Val_Norm": "age_of_residents_norm"})
race_black           = race_black.rename(columns={"Val_Raw": "race_black",                  "Val_Norm": "race_black_norm"})
race_white           = race_white.rename(columns={"Val_Raw": "race_white",                  "Val_Norm": "race_white_norm"})
race_hispanic        = race_hispanic.rename(columns={"Val_Raw": "race_hispanic",            "Val_Norm": "race_hispanic_norm"})
race_asian           = race_asian.rename(columns={"Val_Raw": "race_asian",                  "Val_Norm": "race_asian_norm"})
race_other           = race_other.rename(columns={"Val_Raw": "race_other",                  "Val_Norm": "race_other_norm"})

transit_proximity    = transit_proximity.rename(columns={"Val_Raw": "transit_proximity",    "Val_Norm": "transit_proximity_norm"})
financial_proximity  = financial_proximity.rename(columns={"Val_Raw": "financial_proximity","Val_Norm": "financial_proximity_norm"})
grocery_proximity    = grocery_proximity.rename(columns={"Val_Raw": "grocery_proximity",    "Val_Norm": "grocery_proximity_norm"})

crime_property       = crime_property.rename(columns={"Val_Raw": "crime_property",          "Val_Norm": "crime_property_norm"})
crime_violent        = crime_violent.rename(columns={"Val_Raw": "crime_violent",            "Val_Norm": "crime_violent_norm"})

# Merge all into one dataframe on NPA
qol = household_income
for df in [home_ownership, home_sales_price, rental_cost, rental_houses,
           food_nutrition, health_insurance, employment,
           housing_age, code_violations, foreclosures, new_constructions, housing_density,
           age_of_residents, race_black, race_white, race_hispanic, race_asian, race_other,
           transit_proximity, financial_proximity, grocery_proximity,
           crime_property, crime_violent]:
    qol = pd.merge(qol, df, on="NPA", how="outer")

print(f"Shape: {qol.shape}")
display(qol.head())
qol.describe()

Shape: (458, 49)


,NPA,household_income,household_income_norm,home_ownership,home_ownership_norm,home_sales_price,home_sales_price_norm,rental_cost,rental_cost_norm,rental_houses,...,transit_proximity,transit_proximity_norm,financial_proximity,financial_proximity_norm,grocery_proximity,grocery_proximity_norm,crime_property,crime_property_norm,crime_violent,crime_violent_norm
0,2,None,70647.0,None,38.692098,39557500.0,488364.0,None,1020.0,119.0,...,1136.0,100.000000,278.0,24.471831,689.0,60.651408,175.0,81.661223,16.0,7.466169
1,3,None,108531.0,None,38.901134,267504000.0,667092.0,None,1679.0,297.0,...,9704.0,100.000000,9704.0,100.000000,5863.0,60.418384,695.0,57.338503,42.0,3.465061
2,4,None,250001.0,None,100.000000,52256500.0,1493043.0,None,NaN,29.0,...,354.0,86.552567,62.0,15.158924,73.0,17.848411,9.0,7.705479,0.0,0.000000
3,5,None,49926.0,None,23.175966,4080500.0,255031.0,None,784.0,118.0,...,336.0,96.551724,66.0,18.965517,28.0,8.045977,27.0,34.974093,8.0,10.362694
4,6,None,39202.0,None,31.459170,28079000.0,445698.0,None,NaN,372.0,...,823.0,100.000000,575.0,69.866343,384.0,46.658566,133.0,74.012243,55.0,30.606566


,NPA,household_income_norm,home_ownership_norm,home_sales_price,home_sales_price_norm,rental_cost_norm,rental_houses,rental_houses_norm,food_nutrition_norm,health_insurance,...,transit_proximity,transit_proximity_norm,financial_proximity,financial_proximity_norm,grocery_proximity,grocery_proximity_norm,crime_property,crime_property_norm,crime_violent,crime_violent_norm
count,458.000000,448.000000,455.000000,4.430000e+02,4.430000e+02,392.000000,442.000000,442.000000,455.000000,457.0,...,402.000000,402.000000,286.000000,286.000000,307.000000,307.000000,458.000000,456.000000,458.000000,456.000000
mean,234.862445,89182.459821,57.625811,3.855062e+07,4.748331e+05,1345.576531,139.952489,24.478813,0.192987,0.0,...,923.940299,77.905648,597.342657,43.277092,520.680782,40.664440,76.397380,32.708057,10.174672,4.759897
std,136.110670,45423.664163,27.779995,4.952843e+07,2.715385e+05,386.724840,122.493260,16.248128,0.171913,0.0,...,1038.155258,29.670146,1051.732333,31.175914,711.964925,29.230714,128.647719,47.781295,16.071369,7.729508
min,2.000000,9444.000000,0.000000,4.060000e+05,1.266550e+05,0.000000,1.000000,2.000000,0.000000,0.0,...,2.000000,0.100654,0.000000,0.000000,1.000000,0.084459,0.000000,0.000000,0.000000,0.000000
25%,118.250000,55224.250000,38.779062,1.186000e+07,3.115790e+05,1095.500000,50.000000,12.737578,0.053500,0.0,...,436.250000,64.094793,148.500000,15.362737,173.500000,15.667840,14.250000,7.106504,1.000000,0.363781
50%,233.500000,78064.500000,59.559676,2.455150e+07,3.977910e+05,1293.500000,105.000000,21.446997,0.167000,0.0,...,725.500000,95.033496,398.500000,40.999265,362.000000,35.645933,42.000000,20.127261,5.000000,2.496581
75%,351.750000,110645.000000,81.367322,4.847250e+07,5.399110e+05,1579.750000,202.000000,31.979334,0.290000,0.0,...,1140.250000,100.000000,726.000000,68.067059,643.500000,62.584577,90.000000,42.462291,13.000000,5.996214
max,476.000000,250001.000000,100.000000,5.559040e+08,2.660173e+06,3500.000000,755.000000,100.000000,1.874000,0.0,...,14315.000000,100.000000,12953.000000,100.000000,8459.000000,100.000000,1789.000000,638.381201,190.000000,90.078329


In [24]:
for col in qol.columns:
    none_count = (qol[col].isna() | (qol[col] == "None")).sum()
    print(f"{col}: {none_count}")

NPA: 0
household_income: 458
household_income_norm: 10
home_ownership: 458
home_ownership_norm: 3
home_sales_price: 15
home_sales_price_norm: 15
rental_cost: 458
rental_cost_norm: 66
rental_houses: 16
rental_houses_norm: 16
food_nutrition: 458
food_nutrition_norm: 3
health_insurance: 1
health_insurance_norm: 1
employment: 458
employment_norm: 2
housing_age: 458
housing_age_norm: 14
code_violations: 87
code_violations_norm: 87
foreclosures: 0
foreclosures_norm: 7
new_constructions: 0
new_constructions_norm: 0
housing_density: 0
housing_density_norm: 0
age_of_residents: 458
age_of_residents_norm: 4
race_black: 458
race_black_norm: 3
race_white: 458
race_white_norm: 3
race_hispanic: 458
race_hispanic_norm: 3
race_asian: 458
race_asian_norm: 3
race_other: 458
race_other_norm: 3
transit_proximity: 56
transit_proximity_norm: 56
financial_proximity: 172
financial_proximity_norm: 172
grocery_proximity: 151
grocery_proximity_norm: 151
crime_property: 0
crime_property_norm: 2
crime_violent: 0
cr

In [25]:
# For each variable, use Val_Raw if available, otherwise fall back to Val_Norm
raw_cols = [c for c in qol.columns if not c.endswith("_norm") and c != "NPA"]

for col in raw_cols:
    norm_col = f"{col}_norm"
    if norm_col in qol.columns:
        # If Val_Raw is entirely null, replace with Val_Norm
        if qol[col].isnull().all():
            qol[col] = qol[norm_col]

# Drop all _norm columns since we no longer need them
qol = qol[[c for c in qol.columns if not c.endswith("_norm")]]

print(qol.shape)
print(qol.isnull().sum())
display(qol.head())

(458, 25)
NPA                      0
household_income        10
home_ownership           3
home_sales_price        15
rental_cost             66
rental_houses           16
food_nutrition           3
health_insurance         1
employment               2
housing_age             14
code_violations         87
foreclosures             0
new_constructions        0
housing_density          0
age_of_residents         4
race_black               3
race_white               3
race_hispanic            3
race_asian               3
race_other               3
transit_proximity       56
financial_proximity    172
grocery_proximity      151
crime_property           0
crime_violent            0
dtype: int64


,NPA,household_income,home_ownership,home_sales_price,rental_cost,rental_houses,food_nutrition,health_insurance,employment,housing_age,...,race_black,race_white,race_hispanic,race_asian,race_other,transit_proximity,financial_proximity,grocery_proximity,crime_property,crime_violent
0,2,70647.0,38.692098,39557500.0,1020.0,119.0,0.234,0.0,95.491803,70.0,...,20.91,59.78,10.59,1.73,7.00,1136.0,278.0,689.0,175.0,16.0
1,3,108531.0,38.901134,267504000.0,1679.0,297.0,0.029,0.0,97.642935,73.0,...,6.67,79.00,5.35,4.54,4.45,9704.0,9704.0,5863.0,695.0,42.0
2,4,250001.0,100.000000,52256500.0,NaN,29.0,0.003,0.0,98.041958,45.0,...,1.63,88.44,2.91,4.28,2.74,354.0,62.0,73.0,9.0,0.0
3,5,49926.0,23.175966,4080500.0,784.0,118.0,0.497,0.0,84.722222,61.0,...,71.24,10.75,12.31,1.17,4.53,336.0,66.0,28.0,27.0,8.0
4,6,39202.0,31.459170,28079000.0,NaN,372.0,0.393,0.0,100.000000,67.0,...,70.90,14.64,10.13,0.50,3.84,823.0,575.0,384.0,133.0,55.0


We still have records with NaN in a few variables. Double checked the online tool, and there were no figures reported. To fix this, NPAs with missing values will be imputed with a citywide median.

In [26]:
# Save NPA values before imputing
npa_values = qol["NPA"].values

# Impute everything except NPA
imputer = SimpleImputer(strategy="median")
qol = pd.DataFrame(
    imputer.fit_transform(qol.drop(columns="NPA")),
    columns=qol.drop(columns="NPA").columns
)

# Add NPA back in
qol.insert(0, "NPA", npa_values)

print(qol.isnull().sum().sum())  # should be 0
print(qol.shape)

0
(458, 25)


In [27]:
display(qol.head())

,NPA,household_income,home_ownership,home_sales_price,rental_cost,rental_houses,food_nutrition,health_insurance,employment,housing_age,...,race_black,race_white,race_hispanic,race_asian,race_other,transit_proximity,financial_proximity,grocery_proximity,crime_property,crime_violent
0,2,70647.0,38.692098,39557500.0,1020.0,119.0,0.234,0.0,95.491803,70.0,...,20.91,59.78,10.59,1.73,7.00,1136.0,278.0,689.0,175.0,16.0
1,3,108531.0,38.901134,267504000.0,1679.0,297.0,0.029,0.0,97.642935,73.0,...,6.67,79.00,5.35,4.54,4.45,9704.0,9704.0,5863.0,695.0,42.0
2,4,250001.0,100.000000,52256500.0,1293.5,29.0,0.003,0.0,98.041958,45.0,...,1.63,88.44,2.91,4.28,2.74,354.0,62.0,73.0,9.0,0.0
3,5,49926.0,23.175966,4080500.0,784.0,118.0,0.497,0.0,84.722222,61.0,...,71.24,10.75,12.31,1.17,4.53,336.0,66.0,28.0,27.0,8.0
4,6,39202.0,31.459170,28079000.0,1293.5,372.0,0.393,0.0,100.000000,67.0,...,70.90,14.64,10.13,0.50,3.84,823.0,575.0,384.0,133.0,55.0


#### 'vul_disp'  
Vulnerability to Displacement dataset downloaded as a csv from:  
https://data.charlottenc.gov/datasets/charlotte::vulnerability-to-displacement-by-npa/about   

"As a critical layer of analysis, an overlay identifying concentrations of residents that are vulnerable to being impacted negatively by change.

As a critical layer of analysis, an overlay identifying concentrations of residents that are vulnerable to being impacted negatively by change was developed. The analysis identifies residents that have characteristics that tend to make them more vulnerable to potential displacement. Unfortunately, the same characteristics that make certain populations susceptible to displacement are used in identifying whether environmental impacts are justly distributed. Areas with higher concentrations of vulnerable populations are overlaid on the access to opportunity and environmental justice maps to better understand how physical conditions, access, costs and benefits impact residents that have suffered from systemic racial and other social discrimination and/or are less likely to be able to adapt to economic and other changes. These data have been updated to include tenure and is calculated by NPA. It is used in the Displacement Risk Dashboard."

In [28]:
# Load data
vul_disp = pd.read_csv("https://raw.githubusercontent.com/coffeeNqueries/DTSC2302-Project3/refs/heads/main/data/raw_data/Vulnerability_to_Displacement_by_NPA.csv")

print("=== Vulnerability to Displacement ===")
print(f"Shape: {vul_disp.shape}")
print(vul_disp.dtypes)
print(vul_disp.head())

=== Vulnerability to Displacement ===
Shape: (462, 18)
OBJECTID               int64
ID                     int64
GlobalID              object
NPA                    int64
PctHighSchool         object
PctNonWhite           object
Pct65_                object
PctHH_Poverty         object
PctHome_Ownership     object
HighSchool           float64
NonWhite             float64
F65_                 float64
Poverty              float64
Home_Ownership       float64
Score                float64
Vulnerable            object
ShapeSTArea          float64
ShapeSTLength        float64
dtype: object
   OBJECTID   ID                                GlobalID  NPA PctHighSchool  \
0         1  399  {16B79D3A-75BD-43F8-950A-8E6EA7D15EC2}  399            6%   
1         2  418  {753134D3-45B8-4330-95D1-8E2FA25EC1DC}  418           12%   
2         3  233  {17A6A11C-631C-4160-9627-A52E74DECFAC}  233           16%   
3         4  270  {26EC21F6-3C6C-4A25-BE1D-89F50295E608}  270           51%   
4         5   

Load data into SQLite database

In [ ]:
# Create a connection to a new SQLite database file
conn = sqlite3.connect("../data/charlotte_housing.db")

# Write each dataset to its own table (drop geometry columns since SQLite is not a spatial database)
clt_holc.drop(columns="geometry").to_sql("clt_holc", conn, if_exists="replace", index=False)

npa.drop(columns="geometry").to_sql("npa", conn, if_exists="replace", index=False)

npa_holc.to_sql("npa_holc", conn, if_exists="replace", index=False)

qol.to_sql("qol", conn, if_exists="replace", index=False)

vul_disp.to_sql("vul_disp", conn, if_exists="replace", index=False)

# Verify tables
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("Tables in database:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"  {table[0]}: {count} rows")

# Close connection
conn.close()